# Bouncing Ball Benchmark - Analysis & Visualisation

Analyses two CSV files produced by the C++ benchmark:
- **`benchmark.csv`** - aggregated metrics per (solver, dt)
- **`energy_drift.csv`** - E(t) time series for selected dt values

Plots produced:
1. Peak height error vs dt - convergence order (contact timing dominates)
2. **Flight energy drift vs dt - integrator quality in free flight (new, key result)**
3. Max energy drift vs dt
4. Cost vs precision (ops_total vs height error)
5. CPU time vs dt
6. Bounce count vs dt - solver stability
7. E(t) time series
8. Summary statistics table

In [ ]:
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D
from scipy import stats

plt.rcParams.update(
    {
        "figure.dpi": 130,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "font.size": 11,
    }
)

SOLVER_STYLE = {
    "Euler": {"color": "#E24B4A", "marker": "o", "ls": "-"},
    "Verlet": {"color": "#378ADD", "marker": "s", "ls": "--"},
    "RK4": {"color": "#1D9E75", "marker": "^", "ls": "-."},
}
ABOX = dict(boxstyle="round,pad=0.4", fc="white", ec="#cccccc", alpha=0.92, lw=0.8)

def solver_legend(ax, loc="lower right"):
    handles = [
        Line2D(
            [0],
            [0],
            color=s["color"],
            marker=s["marker"],
            linestyle=s["ls"],
            label=name,
            markersize=6,
        )
        for name, s in SOLVER_STYLE.items()
    ]
    ax.legend(handles=handles, framealpha=0.5, loc=loc)

# Analytical parameters (must match the C++ benchmark)
Z0 = 20.0
RADIUS = 2.0
RESTITUTION = 0.9
G = 9.81

def analytical_peak(n):
    """Analytical peak height after bounce n (1-indexed)."""
    return RADIUS + (Z0 - RADIUS) * RESTITUTION ** (2 * n)

BENCHMARK_CSV = Path("results/benchmark.csv")
ENERGY_DRIFT_CSV = Path("results/energy_drift.csv")

df = pd.read_csv(BENCHMARK_CSV)
ede = pd.read_csv(ENERGY_DRIFT_CSV)

# Convert cpu_us to ms for readability
df["cpu_ms"] = df["cpu_us"] / 1000.0

print(
    f"benchmark.csv    : {len(df)} rows  ({df['solver'].nunique()} solvers, {df['dt'].nunique()} dt values)"
)
print(f"energy_drift.csv : {len(ede)} rows")
df.head()

## Plot 1 - Peak height error vs dt (convergence order)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for solver, grp in df.groupby("solver"):
    s = SOLVER_STYLE[solver]
    grp = grp.sort_values("dt")
    mask = grp["max_height_error"] > 0
    ax.loglog(
        grp.loc[mask, "dt"],
        grp.loc[mask, "max_height_error"],
        color=s["color"],
        marker=s["marker"],
        ls=s["ls"],
        markersize=4,
        linewidth=1.6,
    )

# Reference slopes (visual guides, not fitted to data)
dt_ref = np.array([1e-5, 5e-1])
ax.loglog(dt_ref, 6 * dt_ref**1, "k:", linewidth=1.2)
ax.loglog(dt_ref, 3e-1 * dt_ref**2, "k--", linewidth=1.2)
ax.loglog(dt_ref, 1e4 * dt_ref**4, "k-.", linewidth=1.2)

# Label reference slopes inline
ax.text(3e-1, 2.8, "O(dt¹)", fontsize=8, color="#888", ha="center")
ax.text(2e-1, 3e-1 * (2e-1) ** 2 * 2.8, "O(dt²)", fontsize=8, color="#888", ha="center")
ax.text(4e-2, 1e4 * (4e-2) ** 4 * 2.8, "O(dt⁴)", fontsize=8, color="#888", ha="center")

# Annotations pointing to actual measured curves
# Euler at dt≈1e-2: error ≈ prefactor * dt^0.86 ≈ 0.030
ax.annotate(
    "Euler: O(dt^0.86)\n(contact timing\n+ free-flight drift)",
    xy=(1e-2, 3e-2),
    xytext=(4e-5, 1.5e-1),
    arrowprops=dict(arrowstyle="->", color="#E24B4A", lw=1.2),
    color="#E24B4A",
    fontsize=9,
    bbox=ABOX,
)

# Verlet ≡ RK4 at dt≈1e-2: error ≈ 0.005
ax.annotate(
    "Verlet ≡ RK4: O(dt^0.90)\n(contact timing only;\nflight error at machine ε)",
    xy=(1e-2, 5e-3),
    xytext=(3e-3, 3e-6),
    arrowprops=dict(arrowstyle="->", color="#378ADD", lw=1.2),
    color="#555",
    fontsize=9,
    bbox=ABOX,
)

ax.set_xlabel("Time step  dt  (s)")
ax.set_ylabel("|height_error|  (relative)")
ax.set_title("Peak height error vs dt - convergence order")

handles_s = [
    Line2D([0], [0], color=s["color"], marker=s["marker"], ls=s["ls"], label=k, markersize=6)
    for k, s in SOLVER_STYLE.items()
]
handles_r = [
    Line2D([0], [0], color="k", ls=":", label="O(dt¹) ref."),
    Line2D([0], [0], color="k", ls="--", label="O(dt²) ref."),
    Line2D([0], [0], color="k", ls="-.", label="O(dt⁴) ref."),
]
ax.legend(handles=handles_s + handles_r, fontsize=9, framealpha=0.5, ncol=2, loc="upper left")

plt.tight_layout()
plt.savefig("results/plot_convergence.png", dpi=150, bbox_inches="tight")
plt.show()


## Estimated convergence order (linear regression on log-log)

In [ ]:
print(f"{'Solver':<10} {'Estimated order':>16} {'R²':>8}")
print("-" * 38)

for solver, grp in df.groupby("solver"):
    grp = grp.sort_values("dt")
    mask = (grp["max_height_error"] > 1e-12) & (grp["max_height_error"] > 0)
    if mask.sum() < 4:
        print(f"{solver:<10} {'not enough points':>16}")
        continue
    log_dt = np.log10(grp.loc[mask, "dt"])
    log_err = np.log10(grp.loc[mask, "max_height_error"])
    slope, _, r, *_ = stats.linregress(log_dt, log_err)
    print(f"{solver:<10} {slope:>16.3f} {r**2:>8.4f}")

## Plot 2 - Flight energy drift vs dt (integrator quality, free-flight only)

**What this measures:** energy conservation error *between* bounces - i.e., during the free-flight phases when the only force is constant gravity. Physical energy loss at contact (due to `e = 0.9`) is excluded by resetting the reference after each bounce.

**Why this is the key result:** for a constant-gravity trajectory (quadratic), Störmer-Verlet and RK4 are *exact* - they produce no numerical energy error regardless of dt. Euler drifts as O(dt) because the semi-implicit scheme introduces a `−½ g² dt²` dissipation per step. The log-log plot shows the three regimes clearly:
- **Euler:** clean O(dt) slope
- **Verlet / RK4:** machine-precision floor (~1e-13 with double), indistinguishable

This separates integrator quality from contact-model behaviour in a way that `max_energy_drift` cannot, since that metric conflates physical energy loss (∼73 % after 7 bounces) with numerical error.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for solver, grp in df.groupby("solver"):
    s = SOLVER_STYLE[solver]
    grp = grp.sort_values("dt")
    # Only plot where at least one bounce occurred (flight reference is meaningful)
    mask = (grp["max_flight_energy_drift"] > 0) & (grp["bounce_count"] > 0)
    ax.loglog(
        grp.loc[mask, "dt"],
        grp.loc[mask, "max_flight_energy_drift"],
        color=s["color"],
        marker=s["marker"],
        ls=s["ls"],
        markersize=4,
        linewidth=1.6,
    )

# Reference slope O(dt^1) fitted to Euler
euler = df[df["solver"] == "Euler"].sort_values("dt")
euler_valid = euler[(euler["max_flight_energy_drift"] > 0) & (euler["bounce_count"] > 0)]
if len(euler_valid) >= 4:
    log_dt = np.log10(euler_valid["dt"])
    log_err = np.log10(euler_valid["max_flight_energy_drift"])
    slope, intercept, *_ = stats.linregress(log_dt, log_err)
    dt_fit = np.array([euler_valid["dt"].min(), euler_valid["dt"].max()])
    ax.loglog(
        dt_fit, 10**intercept * dt_fit**slope, "k:", linewidth=1.4, label=f"O(dt^{slope:.2f})"
    )

ax.annotate(
    "Euler: O(dt) numerical\nenergy dissipation\n(−½g²dt² per step)",
    xy=(1e-3, euler_valid["max_flight_energy_drift"].min() * 3),
    xytext=(3e-4, 1e-5),
    arrowprops=dict(arrowstyle="->", color="#E24B4A", lw=1.2),
    color="#E24B4A",
    fontsize=9,
    bbox=ABOX,
)

# Annotate Verlet / RK4 floor
verlet_floor = df[(df["solver"] == "Verlet") & (df["bounce_count"] > 0)][
    "max_flight_energy_drift"
].median()
ax.axhline(verlet_floor, color="#378ADD", ls=":", lw=1.2, alpha=0.6)
ax.text(
    1e-4,
    verlet_floor * 3,
    f"Verlet / RK4 floor ≈ {verlet_floor:.1e}\n(machine precision - exact for quadratic trajectory)",
    color="#378ADD",
    fontsize=8,
    bbox=ABOX,
)

ax.set_xlabel("Time step  dt  (s)")
ax.set_ylabel("Max |ΔE / E_ref|  between bounces")
ax.set_title("Flight energy drift vs dt - integrator quality in free flight")
handles_s = [
    Line2D([0], [0], color=s["color"], marker=s["marker"], ls=s["ls"], label=k, markersize=6)
    for k, s in SOLVER_STYLE.items()
]
ax.legend(handles=handles_s, fontsize=9, framealpha=0.5, loc="upper left")

plt.tight_layout()
plt.savefig("results/plot_flight_energy_drift.png", dpi=150, bbox_inches="tight")
plt.show()

# Print regression for all solvers
print(f"\n{'Solver':<10} {'Flight drift order':>20} {'R²':>8}  (only valid when bounce_count > 0)")
print("-" * 55)
for solver, grp in df.groupby("solver"):
    grp = grp.sort_values("dt")
    mask = (grp["max_flight_energy_drift"] > 1e-15) & (grp["bounce_count"] > 0)
    if mask.sum() < 4:
        print(f"{solver:<10} {'< 4 points (near machine ε)':>20}")
        continue
    log_dt = np.log10(grp.loc[mask, "dt"])
    log_err = np.log10(grp.loc[mask, "max_flight_energy_drift"])
    slope, _, r, *_ = stats.linregress(log_dt, log_err)
    print(f"{solver:<10} {slope:>20.3f} {r**2:>8.4f}")

## Plot 3 - Max energy drift vs dt

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for solver, grp in df.groupby("solver"):
    s = SOLVER_STYLE[solver]
    grp = grp.sort_values("dt")
    mask = grp["max_energy_drift"] > 0
    ax.loglog(
        grp.loc[mask, "dt"],
        grp.loc[mask, "max_energy_drift"],
        color=s["color"],
        marker=s["marker"],
        ls=s["ls"],
        markersize=4,
        linewidth=1.6,
    )

energy_floor = 1.0 - RESTITUTION**2
plateau = df[df["bounce_count"] > 0]["max_energy_drift"].median()

# Plateau line (physical loss from 6 bounces)
ax.axhline(plateau, color="gray", ls=":", lw=1.2, alpha=0.7)
ax.text(
    1.2e-5,
    plateau * 1.4,
    f"Physical plateau ≈ {plateau:.3f}\n(6 bounces, e=0.9 - solver-independent)",
    fontsize=8,
    color="gray",
    bbox=ABOX,
)

# Euler large-dt regime annotation
euler_large = df[(df["solver"] == "Euler") & (df["bounce_count"] == 0)].sort_values("dt")
if len(euler_large) > 0:
    ann_row = euler_large.iloc[len(euler_large) // 2]
    ax.annotate(
        f"Euler (tunneling, dt>0.069 s):\npure free-flight dissipation\n½g²·dt²·N/E₀ - up to 221%",
        xy=(ann_row["dt"], ann_row["max_energy_drift"]),
        xytext=(3e-3, 1.5),
        arrowprops=dict(arrowstyle="->", color="#E24B4A", lw=1.2),
        color="#E24B4A",
        fontsize=9,
        bbox=ABOX,
    )

# Verlet & RK4 plateau annotation
ax.annotate(
    "Verlet & RK4: machine-ε\nflight drift + physical plateau",
    xy=(1e-3, plateau * 0.92),
    xytext=(3e-4, plateau * 0.1),
    arrowprops=dict(arrowstyle="->", color="#378ADD", lw=1.2),
    color="#555",
    fontsize=9,
    bbox=ABOX,
)

ax.set_xlabel("Time step  dt  (s)")
ax.set_ylabel("|E(t) - E₀| / E₀  (max over simulation)")
ax.set_title("Max energy drift vs dt")
solver_legend(ax, loc="lower right")

plt.tight_layout()
plt.savefig("results/plot_energy_drift_vs_dt.png", dpi=150, bbox_inches="tight")
plt.show()


## Plot 4 - Cost vs precision

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for solver, grp in df.groupby("solver"):
    s = SOLVER_STYLE[solver]
    grp = grp.sort_values("ops_total")
    mask = grp["max_height_error"] > 0
    ax.loglog(
        grp.loc[mask, "ops_total"],
        grp.loc[mask, "max_height_error"],
        color=s["color"],
        marker=s["marker"],
        ls=s["ls"],
        markersize=4,
        linewidth=1.6,
    )

# Crossover zone
ax.axvspan(1e4, 1e5, alpha=0.07, color="gray")
ax.text(3e4, 3e-3, "crossover\nzone", fontsize=8, color="gray", ha="center")

# Annotate Euler (always dominated)
euler_mid = (
    df[(df["solver"] == "Euler") & (df["max_height_error"] > 0)].sort_values("ops_total").iloc[-3]
)
ax.annotate(
    "Euler: always dominated\n(dual O(dt) error source)",
    xy=(euler_mid["ops_total"], euler_mid["max_height_error"]),
    xytext=(5e5, 2e-1),
    arrowprops=dict(arrowstyle="->", color="#E24B4A", lw=1.2),
    color="#E24B4A",
    fontsize=9,
    bbox=ABOX,
)

# Annotate Verlet
vlt_mid = (
    df[(df["solver"] == "Verlet") & (df["max_height_error"] > 0)].sort_values("ops_total").iloc[-5]
)
ax.annotate(
    "Verlet: best at high-budget\n(same error as RK4, ×4 cheaper)",
    xy=(vlt_mid["ops_total"], vlt_mid["max_height_error"]),
    xytext=(5e5, 5e-4),
    arrowprops=dict(arrowstyle="->", color="#378ADD", lw=1.2),
    color="#378ADD",
    fontsize=9,
    bbox=ABOX,
)

# Annotate RK4
rk4_mid = (
    df[(df["solver"] == "RK4") & (df["max_height_error"] > 0)].sort_values("ops_total").iloc[5]
)
ax.annotate(
    "RK4: best at low-budget / coarse dt",
    xy=(rk4_mid["ops_total"], rk4_mid["max_height_error"]),
    xytext=(3e2, 5e-5),
    arrowprops=dict(arrowstyle="->", color="#1D9E75", lw=1.2),
    color="#1D9E75",
    fontsize=9,
    bbox=ABOX,
)

ax.set_xlabel("Total force evaluations  (ops_total)  →  more expensive")
ax.set_ylabel("|height_error|  (relative)")
ax.set_title("Cost vs precision - iso-cost comparison")
ax.invert_xaxis()
solver_legend(ax, loc="upper right")

plt.tight_layout()
plt.savefig("results/plot_cost_vs_precision.png", dpi=150, bbox_inches="tight")
plt.show()


## Plot 4_bis - CPU time vs dt

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for solver, grp in df.groupby("solver"):
    s = SOLVER_STYLE[solver]
    grp = grp.sort_values("dt")
    ax.loglog(
        grp["dt"],
        grp["cpu_ms"],
        color=s["color"],
        marker=s["marker"],
        ls=s["ls"],
        markersize=4,
        linewidth=1.6,
    )

# Compute cost ratios at median dt
mid_dt = df["dt"].median()
cpu_mid = {}
for solver, grp in df.groupby("solver"):
    row = grp.iloc[(grp["dt"] - mid_dt).abs().argsort().iloc[:1]]
    cpu_mid[solver] = float(row["cpu_ms"].iloc[0])

rk4_ratio = cpu_mid["RK4"] / cpu_mid["Euler"]
vlt_ratio = cpu_mid["Verlet"] / cpu_mid["Euler"]

# Ratio annotations at mid-dt
mid_row_e = df[(df["solver"] == "Euler")].iloc[
    (df[df["solver"] == "Euler"]["dt"] - mid_dt).abs().argsort().iloc[:1]
]
mid_row_v = df[(df["solver"] == "Verlet")].iloc[
    (df[df["solver"] == "Verlet"]["dt"] - mid_dt).abs().argsort().iloc[:1]
]
mid_row_r = df[(df["solver"] == "RK4")].iloc[
    (df[df["solver"] == "RK4"]["dt"] - mid_dt).abs().argsort().iloc[:1]
]

ax.annotate(
    f"RK4 / Euler ≈ {rk4_ratio:.1f}×\n(theory: 4× - shared per-step\noverhead shrinks gap)",
    xy=(float(mid_row_r["dt"].iloc[0]), float(mid_row_r["cpu_ms"].iloc[0])),
    xytext=(3e-4, 3e4),
    arrowprops=dict(arrowstyle="->", color="#1D9E75", lw=1.2),
    color="#1D9E75",
    fontsize=9,
    bbox=ABOX,
)

ax.annotate(
    f"Verlet / Euler ≈ {vlt_ratio:.1f}×\n(x_prev storage + central\ndifference velocity update)",
    xy=(float(mid_row_v["dt"].iloc[0]), float(mid_row_v["cpu_ms"].iloc[0])),
    xytext=(3e-3, 1e3),
    arrowprops=dict(arrowstyle="->", color="#378ADD", lw=1.2),
    color="#378ADD",
    fontsize=9,
    bbox=ABOX,
)

ax.annotate(
    "All solvers: O(1/dt)\n- halving dt doubles run time",
    xy=(float(mid_row_e["dt"].iloc[0]), float(mid_row_e["cpu_ms"].iloc[0])),
    xytext=(1e-2, 2e1),
    arrowprops=dict(arrowstyle="->", color="#E24B4A", lw=1.2),
    color="#555",
    fontsize=9,
    bbox=ABOX,
)

ax.set_xlabel("Time step  dt  (s)")
ax.set_ylabel("CPU time  (ms)")
ax.set_title("CPU time vs dt")
solver_legend(ax, loc="upper right")

plt.tight_layout()
plt.savefig("results/plot_cpu_time.png", dpi=150, bbox_inches="tight")
plt.show()


## Plot 5 - Bounce count vs dt (solver stability)

In [ ]:
def expected_bounces(total_time, z0=Z0, r=RADIUS, e=RESTITUTION, g=G):
    """Analytical bounce count for a given simulation duration."""
    h0_eff = z0 - r
    t = np.sqrt(2 * h0_eff / g)  # time to first contact
    n = 1  # count the 1st contact
    while True:
        h_post = h0_eff * e ** (2 * n)  # height after nth bounce
        flight = 2 * np.sqrt(2 * h_post / g)
        if t + flight > total_time:
            break
        t += flight
        n += 1
    return n

expected_n = expected_bounces(18.0)
print(f"Analytical expected bounces in 18 s: {expected_n}")

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for solver, grp in df.groupby("solver"):
    s = SOLVER_STYLE[solver]
    grp = grp.sort_values("dt")
    ax.semilogx(
        grp["dt"],
        grp["bounce_count"],
        color=s["color"],
        marker=s["marker"],
        ls=s["ls"],
        markersize=4,
        linewidth=1.6,
    )

ax.axhline(expected_n, color="black", ls=":", lw=1.5, alpha=0.7)
ax.text(
    6e-1,
    expected_n + 0.3,
    f"Analytical: {expected_n} bounces (18 s)",
    fontsize=9,
    ha="right",
    color="black",
)

# Euler tunneling threshold annotation
euler_tun_dt = df[(df["solver"] == "Euler") & (df["bounce_count"] == 0)]["dt"].min()
ax.annotate(
    f"Euler tunnels\ndt > {euler_tun_dt:.3f} s",
    xy=(euler_tun_dt, 0.2),
    xytext=(euler_tun_dt * 0.2, 2.5),
    arrowprops=dict(arrowstyle="->", color="#E24B4A", lw=1.2),
    color="#E24B4A",
    fontsize=9,
    bbox=ABOX,
)

# Verlet/RK4 tunneling threshold annotation
vlt_tun_dt = df[(df["solver"] == "Verlet") & (df["bounce_count"] == 0)]["dt"].min()
ax.annotate(
    f"Verlet / RK4 tunnel\ndt > {vlt_tun_dt:.3f} s",
    xy=(vlt_tun_dt, 0.2),
    xytext=(vlt_tun_dt * 2, 1.5),
    arrowprops=dict(arrowstyle="->", color="#378ADD", lw=1.2),
    color="#555",
    fontsize=9,
    bbox=ABOX,
)

ax.set_xlabel("Time step  dt  (s)")
ax.set_ylabel("Bounce count  (detected)")
ax.set_title("Bounce count vs dt - solver stability (18 s simulation)")
solver_legend(ax, loc="lower left")

plt.tight_layout()
plt.savefig("results/plot_bounce_count.png", dpi=150, bbox_inches="tight")
plt.show()


## Plot 6 - E(t) time series

In [ ]:
selected_dts = sorted(ede["dt"].unique())
n_panels = len(selected_dts)

fig = plt.figure(figsize=(15, 3.5 * n_panels))
gs = gridspec.GridSpec(n_panels, 1, hspace=0.5)

for i, dt_val in enumerate(selected_dts):
    ax = fig.add_subplot(gs[i, 0])
    for solver in ["Euler", "Verlet", "RK4"]:
        sub = ede[(ede["solver"] == solver) & np.isclose(ede["dt"], dt_val)].sort_values("time")
        s = SOLVER_STYLE[solver]
        ax.semilogy(
            sub["time"],
            sub["energy_drift"].clip(lower=1e-17),
            color=s["color"],
            ls=s["ls"],
            lw=1.4,
            label=solver,
            alpha=0.85,
        )
    ax.set_title(f"Energy drift  E(t)  -  dt = {dt_val:.2e} s", fontsize=10)
    ax.set_ylabel("|E(t) − E₀| / E₀", fontsize=9)
    ax.set_xlabel("Time (s)", fontsize=9)
    solver_legend(ax, loc="upper left")

fig.suptitle("E(t) time series - energy drift over simulation", fontsize=12, y=1.005)
plt.savefig("results/plot_energy_time_series.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
rows = []
for solver, grp in df.groupby("solver"):
    best_row = grp.loc[grp["dt"].idxmin()]
    worst_row = grp.loc[grp["dt"].idxmax()]

    # Bounce count: accept ±1 tolerance (discrete timing never gives exactly N)
    stable_pct = (abs(grp["bounce_count"] - expected_n) <= 1).mean() * 100

    rows.append(
        {
            "Solver": solver,
            "Best height error": f"{best_row['max_height_error']:.2e}",
            "Flight energy drift (min dt)": f"{best_row['max_flight_energy_drift']:.2e}",
            "Max total energy drift": f"{best_row['max_energy_drift']:.2e}",
            "Bounce count ±1 (%)": f"{stable_pct:.0f}%",
            "CPU @ min dt (ms)": f"{best_row['cpu_ms']:.1f}",
        }
    )

summary = pd.DataFrame(rows).set_index("Solver")
print("Notes:")
print(
    f"  • 'Flight energy drift' = integrator error only (between bounces). Verlet/RK4 are at machine ε."
)
print(
    f"  • 'Max total energy drift' at min dt ≈ 1 − e^(2×7) = {1 - RESTITUTION ** (2 * 7):.3f} (7 bounces × physical loss)"
)
print(
    f"  • 'Bounce count ±1' uses |count − {expected_n}| ≤ 1 (discrete timing offset of ±1 bounce is expected)\n"
)
summary

## Summary statistics table

In [ ]:

plot_files = [
    "plot_convergence.png",
    "plot_flight_energy_drift.png",
    "plot_energy_drift_vs_dt.png",
    "plot_cost_vs_precision.png",
    "plot_cpu_time.png",
    "plot_bounce_count.png",
    "plot_energy_time_series.png",
]

with PdfPages("results/benchmark_report.pdf") as pdf:
    for path in plot_files:
        try:
            img = plt.imread("results/" + path)
            fig, ax = plt.subplots(figsize=(14, 7))
            ax.imshow(img)
            ax.axis("off")
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
        except FileNotFoundError:
            print(f"Skipping {path} (not yet generated - run all cells first)")

print("PDF saved to benchmark_report.pdf")

## Export all plots to PDF

In [ ]:

plot_files = [
    "plot_convergence.png",
    "plot_energy_drift_vs_dt.png",
    "plot_cost_vs_precision.png",
    "plot_cpu_time.png",
    "plot_bounce_count.png",
    "plot_energy_time_series.png",
]

with PdfPages("results/benchmark_report.pdf") as pdf:
    for path in plot_files:
        img = plt.imread("results/" + path)
        fig, ax = plt.subplots(figsize=(14, 7))
        ax.imshow(img)
        ax.axis("off")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print("PDF saved to benchmark_report.pdf")